In [ ]:
library(circlize)
library(RColorBrewer)
library(ComplexHeatmap)
library(tidyverse)
library(reshape2)

In [ ]:
## Heatmap related to Fig. 1e

In [ ]:
inmat <- read.csv("/data/work/pathologicalRegion/cellDensity/main_region_cell.stat.txt",sep = "\t")

In [ ]:
dfwilcox2others <- function(df,testcol,compareList){
  levels <- unique(df[,testcol])
  outlist <- data.frame()
  for(i in compareList){
    for(j in  levels){
      test <- df[df[[testcol]]==j,i] %>% as.numeric
      other <- df[df[[testcol]]!=j,i] %>% as.numeric
      res <- wilcox.test(test,other)
      p <- res$p.value
      outlist <- rbind(outlist,c(i,j,p))
    }
    
  }
  outlist <- data.frame(outlist)
  colnames(outlist) <-c("Type","cols","pvalue")
  outlist$pvalue <- as.numeric( outlist$pvalue)
  return(outlist)
}

comlist <- c("Epithelial","Endothelial","Myofibroblast","Plasma","Myeloid","T","B")
pvalue_test <- dfwilcox2others(inmat,"region",comlist)

In [ ]:
p_to_significance_vectorized <- function(p_values) {
  case_when(
    is.na(p_values) ~ NA_character_,
    p_values < 0.001 ~ "***",
    p_values < 0.01 ~ "**",
    p_values < 0.05 ~ "*",
    TRUE ~ ""
  )
}

pvalue_test$psig <- p_to_significance_vectorized(pvalue_test$pvalue)

pvalue_test <- pvalue_test[,c("Type","cols","psig")]

median_valuedf <- inmat %>% group_by(region) %>% 
  summarize(across(comlist,median)) %>% 
  ungroup() %>%
  data.frame %>% melt

colnames(median_valuedf) <- c("region","cell","cell_density")

scaled_median_valuedf <- median_valuedf %>% group_by(cell) %>%
  mutate(cell_density = scale(cell_density)) %>%
  ungroup() %>%
  data.frame
  
########### plot
colnames(pvalue_test) <- c("cell","region","pvalue")
forplot <- merge(scaled_median_valuedf,pvalue_test,by = c("region","cell"))

p <- ggplot(forplot, aes(cell , region)) + 
  geom_tile(aes(fill = cell_density), colour = "white", size = 1)+
  scale_fill_gradientn(
    colors = rev(c("#57121d", "#d56e5e", "#eaebea", "#5390b5", "#1f294e")),
    values = scales::rescale(c(min(forplot$cell_density), mean(forplot$cell_density), max(forplot$cell_density))),
    limits = c(min(forplot$cell_density), max(forplot$cell_density))
  )+
  geom_text(aes(label=pvalue),col ="white",size = 7,fontface = "bold" ) +
  theme_minimal() + 
  theme(axis.title.x=element_blank(), 
        axis.ticks.x=element_blank(), 
        axis.title.y=element_blank(), 
        axis.text.x = element_text(angle = 45, hjust = 1, size = 14), 
        axis.text.y = element_text(size = 14),
        panel.grid = element_blank() )  

ggsave("/data/work/pathologicalRegion/cellDensity/heatmap/cell_density4region.pdf", p, width = 5.5, height = 2.8)

In [ ]:
## Heatmap related to Fig. 2a

In [ ]:
type_csv<-read.table('/data/work/file/sample_type_RR.csv',sep=',', header=TRUE)
type_csv <- subset(type_csv, !(Region %in% c('Filter')))
type_csv[['cluster_names']]<-paste0(type_csv[['sample']],':res',type_csv[['selected_res']],':',type_csv[['Region']],':cluster',type_csv[['selected_cluster']])

In [ ]:
Region_colors <- c(
    'Immune_hot'='#E89A5B', 'Stroma'='#D8C39F', 'IG'='#7FA6D9'
)
Response_colors <- c(
    'PR'='#7788B1', 'SD'='#DD7334'
)
Type_colors <- c(
    'ATR'='#4F6FA8', 'RTR'='#6FB98F', 'RTR/ATR'='#E0C36C'
    
)

m = "ward.D"
ifmaxv<-0
maxv<-0.25
heatmap_path<-'/data/work/STAGATE/heatmap'
region_path<- '/data/work/STAGATE/heatmap/subcluster'

row_order_file<-read.csv(paste0(region_path, '/cluster_order.csv'))
row_order <- row_order_file[, 1]
col_order_file<-read.csv(paste0(region_path, '/cluster_order.csv'))
col_order <- col_order_file[, 1]

In [ ]:
top_n <- 50
csv_path<-paste0(heatmap_path,'/top',top_n,'_heatmap_jaccard.csv')
df<-read.csv(csv_path,row.names=1,check.names = FALSE)
df[is.na(df)]<-1
df <- df[row_order, ]
df <- df[, col_order]
matrix <- as.matrix(df) 
matrix <- matrix[row_order, col_order]
row_Region <- type_csv$Region[match(rownames(df), type_csv$cluster_names)]
row_Response <- type_csv$Response[match(rownames(df), type_csv$cluster_names)]
row_Type <- type_csv$Type[match(rownames(df), type_csv$cluster_names)]
    
column_Region <- type_csv$Region[match(colnames(df), type_csv$cluster_names)]
column_Response <- type_csv$Response[match(colnames(df), type_csv$cluster_names)]
column_Type <- type_csv$Type[match(colnames(df), type_csv$cluster_names)]
    
row_annotation_Region <- rowAnnotation(Region = row_Region, col = list(Region = Region_colors),show_annotation_name = FALSE,
                                          annotation_legend_param = list(
                                                    title = "Region",
                                                    title_gp = gpar(fontsize = 16), 
                                                    labels_gp = gpar(fontsize = 14), 
                                                    legend_height = unit(6, "cm"),
                                                    legend_width = unit(3, "cm")
                                                )
                                          )
row_annotation_Response <- rowAnnotation(Response = row_Response, col = list(Response = Response_colors),show_annotation_name = FALSE,
                                            annotation_legend_param = list(
                                                    title = "Response",
                                                    title_gp = gpar(fontsize = 16), 
                                                    labels_gp = gpar(fontsize = 14), 
                                                    legend_height = unit(6, "cm"),
                                                    legend_width = unit(3, "cm")
                                                )
                                            )
row_annotation_Type <- rowAnnotation(Type = row_Type, col = list(Type = Type_colors),show_annotation_name = FALSE,
                                            annotation_legend_param = list(
                                                    title = "Type",
                                                    title_gp = gpar(fontsize = 16), 
                                                   labels_gp = gpar(fontsize = 14), 
                                                    legend_height = unit(6, "cm"),
                                                    legend_width = unit(3, "cm")
                                                )
                                            )
column_annotation_Region <- HeatmapAnnotation(Region = column_Region, col = list(Region = Region_colors),show_legend = FALSE)
column_annotation_Response <- HeatmapAnnotation(Response = column_Response, col = list(Response = Response_colors),show_legend = FALSE)
column_annotation_Type <- HeatmapAnnotation(Type = column_Type, col = list(Type = Type_colors),show_legend = FALSE)
    
combined_row_annotation <- c(row_annotation_Region, row_annotation_Response,row_annotation_Type)
combined_column_annotation <- c(column_annotation_Region, column_annotation_Response,column_annotation_Type)
    
if(ifmaxv==1){
    maxv <- max(df, na.rm = TRUE)
    vamx_path <- paste0(region_path,'/vmax')
    if (!dir.exists(vamx_path)) {
            dir.create(vamx_path, recursive = TRUE)
    }
    pdf_path<-paste0(region_path,'/vmax/top',top_n,'_heatmap.pdf')
}else{
    maxv<-maxv
    vamx_path <- paste0(region_path,'/vmax',maxv)
if (!dir.exists(vamx_path)) {
        dir.create(vamx_path, recursive = TRUE)
}
    pdf_path<-paste0(region_path,'/vmax',maxv,'/top',top_n,'_heatmap_',maxv,'.pdf')
}

h = Heatmap(matrix,
        col = colorRamp2(c(0,0.3*maxv,0.5*maxv,0.6*maxv,0.7*maxv,0.8*maxv,0.9*maxv,maxv), 
                          c('#FFFFFF','#fb9a06','#cf4446','#a52c60','#781c6d','#4b0c6b','#1b0c42','#000004')),
        clustering_method_rows = m, 
        clustering_method_columns = m, 
        cluster_rows = FALSE,
        row_dend_reorder = FALSE,
        column_dend_reorder = FALSE,
        cluster_columns = FALSE,
        show_row_names = TRUE,
        show_column_names = TRUE,
        column_names_side = "bottom",
        column_dend_side = "bottom",
        row_names_side = "left",
        row_dend_side = "left",
        show_column_dend = FALSE,
        show_row_dend = FALSE,
        heatmap_legend_param = list(
            title = "Jaccard index",
            title_gp = gpar(fontsize = 16),
            labels_gp = gpar(fontsize = 14),
            legend_height = unit(6, "cm"),
            legend_width = unit(3, "cm")
        ),
        row_names_gp = gpar(fontsize = 8),
        column_names_gp = gpar(fontsize = 8),
        use_raster = FALSE,
        top_annotation = combined_column_annotation,
        right_annotation = combined_row_annotation,
        width = unit(26, "cm"),
        height = unit(26, "cm"),
        row_dend_width = unit(4, "cm"),
        column_dend_height = unit(4, "cm")
    )
    
row_order_result <- row_order(h)
heatmap_row_names <- rownames(df)[row_order_result]
column_order_result <- column_order(h)
heatmap_col_names <- colnames(df)[column_order_result]
write.csv(heatmap_row_names, file = paste0(vamx_path,'/row_order_top', top_n, '.csv'))
write.csv(heatmap_col_names, file = paste0(vamx_path, '/column_order_top', top_n, '.csv'))

pdf(pdf_path, width = 24, height = 18)
draw(h)
dev.off()

In [ ]:
## Heatmap related to Extended Data Fig. 4c

In [ ]:
type_csv<-read.table('/data/work/file/sample_type.csv',sep=',', header=TRUE)
type_csv[['cluster_names']]<-paste0(type_csv[['sample']],':res',type_csv[['selected_res']],':',type_csv[['Region']])
type_csv$Region <- ifelse(type_csv$Region %in% c('ROI_RR', 'ROI_ATR', 'ROI_RTR', 'ROI_FR'), type_csv$Region, 'other')

In [ ]:
Region_colors<-c(
    'ROI_RR'='#a7474c', 'ROI_ATR'='#68cdda', 'ROI_RTR'='#F5DEB3', 'ROI_FR'='#cdb3d3', 
    'other'= '#CFCFCF'
)
Response_colors <- c(
    'PR'='#DD7334', 'SD'='#7788B1'
)

m = "ward.D"
ifmaxv<-0
maxv<-0.25
heatmap_path<-'/data/work/STAGATE/heatmap'
region_path<- '/data/work/STAGATE/heatmap/4_region'

In [ ]:
top_n <- 100
csv_path<-paste0(heatmap_path,'/top',top_n,'_heatmap_jaccard.csv')
df<-read.csv(csv_path,row.names=1,check.names = FALSE)
df[is.na(df)]<-1
matrix <- as.matrix(df) 
row_Region <- type_csv$Region[match(rownames(df), type_csv$cluster_names)]
row_Response <- type_csv$Response[match(rownames(df), type_csv$cluster_names)]

column_Region <- type_csv$Region[match(colnames(df), type_csv$cluster_names)]
column_Response <- type_csv$Response[match(colnames(df), type_csv$cluster_names)]

row_annotation_Region <- rowAnnotation(Region = row_Region, col = list(Region = Region_colors),show_annotation_name = FALSE,
                                        annotation_legend_param = list(
                                                title = "Region",
                                                title_gp = gpar(fontsize = 16), 
                                                labels_gp = gpar(fontsize = 14), 
                                                legend_height = unit(6, "cm"),
                                                legend_width = unit(3, "cm")
                                            )
                                        )
row_annotation_Response <- rowAnnotation(Response = row_Response, col = list(Response = Response_colors),show_annotation_name = FALSE,
                                        annotation_legend_param = list(
                                                title = "Response",
                                                title_gp = gpar(fontsize = 16), 
                                                labels_gp = gpar(fontsize = 14), 
                                                legend_height = unit(6, "cm"),
                                                legend_width = unit(3, "cm")
                                            )
                                        )
   
column_annotation_Region <- HeatmapAnnotation(Region = column_Region, col = list(Region = Region_colors),show_legend = FALSE)
column_annotation_Response <- HeatmapAnnotation(Response = column_Response, col = list(Response = Response_colors),show_legend = FALSE)
    
combined_row_annotation <- c(row_annotation_Region, row_annotation_Response)
combined_column_annotation <- c(column_annotation_Region, column_annotation_Response)
    
if(ifmaxv==1){
    maxv <- max(df, na.rm = TRUE)
    vamx_path <- paste0(region_path,'/vmax')
    if (!dir.exists(vamx_path)) {
            dir.create(vamx_path, recursive = TRUE)
    }
    pdf_path<-paste0(region_path,'/vmax/top',top_n,'_heatmap.pdf')
}else{
    maxv<-maxv
    vamx_path <- paste0(region_path,'/vmax',maxv)
    if (!dir.exists(vamx_path)) {
            dir.create(vamx_path, recursive = TRUE)
    }
    pdf_path<-paste0(region_path,'/vmax',maxv,'/top',top_n,'_heatmap_',maxv,'.pdf')
}
h = Heatmap(matrix,
        col = colorRamp2(c(0,0.3*maxv,0.5*maxv,0.6*maxv,0.7*maxv,0.8*maxv,0.9*maxv,maxv), 
                          c('#FFFFFF','#fb9a06','#cf4446','#a52c60','#781c6d','#4b0c6b','#1b0c42','#000004')),
        clustering_method_rows = m, 
        clustering_method_columns = m, 
        cluster_rows = TRUE,
        cluster_columns = TRUE,        
        row_dend_reorder = FALSE,
        column_dend_reorder = FALSE,
        show_row_names = TRUE,
        show_column_names = TRUE,
        column_names_side = "bottom",
        column_dend_side = "bottom",
        row_names_side = "left",
        row_dend_side = "left",
        show_column_dend = FALSE,
        show_row_dend = FALSE,
        heatmap_legend_param = list(
            title = "Jaccard index",
            title_gp = gpar(fontsize = 16),
            labels_gp = gpar(fontsize = 14),
            legend_height = unit(6, "cm"),
            legend_width = unit(3, "cm")
        ),
        row_names_gp = gpar(fontsize = 8),
        column_names_gp = gpar(fontsize = 8),
        use_raster = FALSE,
        top_annotation = combined_column_annotation,
        right_annotation = combined_row_annotation,
        width = unit(26, "cm"),
        height = unit(26, "cm"),
        row_dend_width = unit(4, "cm"),
        column_dend_height = unit(4, "cm")
    )
    
    
row_order_result <- row_order(h)
heatmap_row_names <- rownames(df)[row_order_result]
column_order_result <- column_order(h)
heatmap_col_names <- colnames(df)[column_order_result]
write.csv(heatmap_row_names, file = paste0(vamx_path,'/row_order_top', top_n, '.csv'))
write.csv(heatmap_col_names, file = paste0(vamx_path, '/column_order_top', top_n, '.csv'))

pdf(pdf_path, width = 24, height = 18)
draw(h)
dev.off()